# Natural-Text Located-In Absent-Relation Corpus — Demo

**Artifact:** *Natural-Text Located-In Absent-Relation Corpus from Re-DocRED + DocRED*

A document-level **geographic / administrative containment** (`located-in`) corpus over
genuinely-natural Wikipedia introductory prose (Re-DocRED, the completeness-corrected
re-annotation of DocRED). It is the **structural twin** of the natural-kinship corpus and a
**second genuinely-natural absent-relation domain**, built to show that the
*closure-certificate confidence-blindness diagnostic* is **not kinship-specific**.

Each example is **one Wikipedia document**: `input` = detokenized prose, `output` =
`json.dumps(gold_graph)` with `nodes`, `atomic_edges`, `query_edges`,
`absent_relation_pairs`, `contradiction_pairs`.

**Three strata**
1. **Atomic `located_in` edges** — the KB / proof chain (LOC–LOC, deduped).
2. **PRESENT deduction-required queries** (gold *certain*): `never_annotated` (pair was never a
   direct edge — provably non-circular, rare) and `held_out` (a directly-annotated edge that is
   *also* derivable via an alternative ≥2-hop path; the consumer **drops the single direct edge**
   then the engine deduces it — sound because removing a redundant edge preserves the closure).
3. **ABSENT no-derivation pairs** — two regimes: `different_component` (unrelated places) and
   `same_component_sibling` (co-component, neither inside the other).

**What this demo does.** It loads the corpus and runs the **closure engine the dataset is built
for** — `kinship.py`'s forward least-fixpoint UNION closure, reused **verbatim** and parameterized
by a degenerate single-relation transitive table (`located_in ∘ located_in = located_in`). It
reproduces the dataset's **round-trip gate**: every PRESENT query → singleton **EMIT**
(`{located_in}`), every ABSENT pair → **EMPTY in both directions ⇒ ABSTAIN** (the engine never
invents a containment). It also renders a **human-auditable derivation trace-graph**.

> The corpus itself was built with **$0 LLM spend** (pure deterministic data construction).

In [ ]:
# --- Install dependencies (works on Colab AND local Jupyter) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# The engine is pure stdlib (collections, typing). Only matplotlib is needed for the
# final visualization. matplotlib is PRE-INSTALLED on Colab -> install locally only,
# at Colab's exact version, so the local env matches Colab.
if 'google.colab' not in sys.modules:
    _pip('matplotlib==3.10.0')

In [ ]:
# --- Imports ---
import json
from collections import Counter
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
# --- Data loading helper: GitHub raw URL with local fallback (Colab-compatible) ---
import os
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-40a89b-no-derivation-no-relation-a-closure-cert/main/round-7/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
# --- Load the curated demo slice (30 diverse re-docred documents) ---
data = load_data()

meta = data["metadata"]
print("corpus:", meta["name"])
for d in data["datasets"]:
    print(f"  dataset '{d['dataset']}': {len(d['examples'])} demo documents")

# compact per-document overview
print("\n{:42s} {:>5s} {:>4s} {:>4s}  {}".format("doc_id", "edges", "qry", "abs", "strata"))
for d in data["datasets"]:
    for r in d["examples"]:
        g = json.loads(r["output"])
        qk = "+".join(sorted({q["query_kind"] for q in g["query_edges"]})) or "-"
        rs = "+".join(sorted({p["reason"][:4] for p in g["absent_relation_pairs"]})) or "-"
        print("{:42s} {:>5d} {:>4d} {:>4d}  q[{}] a[{}]".format(
            g["doc_id"][:42], len(g["atomic_edges"]), len(g["query_edges"]),
            len(g["absent_relation_pairs"]), qk, rs))

## Configuration

All tunable parameters live here. They start at small demo values — the demo runs the closure
round-trip over the loaded documents. The **full corpus build** runs the identical engine over
**2,604 re-docred + 2,080 docred** documents (see `metadata.build_stats`).

In [ ]:
# ---- Demo configuration ----
MAX_DOCS = 30          # documents (per dataset) to run the closure round-trip gate over.
                       # demo slice has 30; full build = 2604 re-docred + 2080 docred docs.
PRIM = "located_in"    # the single containment primitive the engine emits.
TRACE_MAX_STEPS = 60   # cap on derivation-trace reconstruction steps (kinship.py default).

## The closure engine (`kinship.py`, reused verbatim)

The symbolic half of the pipeline. Below is `kinship.py` **exactly as shipped** (only the
file's `__main__` self-test was removed). It implements the **forward least-fixpoint UNION
derivation** — the sound closure for a finite composition table:

- seed every atomic edge `a→b:t` (and its converse `b→a:conv(t)`);
- close: for every `a–b–c`, if `rules[t1][t2]=t3` is **defined**, add `t3` to `D[(a,c)]`;
- **output contract:** `|D[query]|==1` → **EMIT**; `|D|==0` → **ABSTAIN** (no path / absent);
  `|D|>1` → Mode-B conflict.

For *this* corpus the same engine is driven by a **degenerate transitive table**
(`located_in ∘ located_in = located_in`, `contains ∘ contains = contains`, everything else
undefined). Undefined compositions add nothing → **sound**: sibling/unrelated places derive
**EMPTY** and the engine abstains rather than hallucinate a containment.

In [ ]:
#!/usr/bin/env python3
"""Kinship finite-composition closure engine (the symbolic half of the pipeline).

CLUTRR's kinship calculus is a FINITE COMPOSITION TABLE over 11 abstract relation
TYPES -- and, as the dataset card states verbatim, it is NOT a full relation algebra:
"no general intersection/converse closure beyond these rules". A naive port of the
temporal PC-2 engine (Mackworth converse-intersection path-consistency) is therefore
UNSOUND here -- closing the converse constraints makes ~13% of GOLD-clean chains
spuriously collapse to EMPTY, because two derivation orders can yield two different
(individually valid) relations that the table does not reconcile. (We verified this
directly: PC-2-style closure gives 100% accuracy WHEN it answers but only 0.87
singleton-rate, with 2153/16131 clean rows collapsing.)

The SOUND closure for a finite composition table is a FORWARD LEAST-FIXPOINT
*UNION* DERIVATION over DEFINED compositions only -- exactly the backward/forward
chaining that produces CLUTRR's own `gold_proof`, and exactly what the emitted Prolog
`derive/3` predicate computes:

  D[(a,b)] = { t : t derivable for directed pair a->b by composing atomic links }

  * seed: every atomic edge a->b:t adds t to D[(a,b)] and conv(t) to D[(b,a)];
  * close: while changing, for every a-b-c, for t1 in D[(a,b)], t2 in D[(b,c)],
    if rules[t1][t2]=t3 is DEFINED, add t3 to D[(a,c)] (and conv(t3) to D[(c,a)]);
    UNDEFINED compositions add nothing (SOUND: "unknown", never a wrong fact).

OUTPUT CONTRACT (disjunction-preserving Mode-A / abstain-on-collapse Mode-B):
  * |D[query]| == 1  -> EMIT the relation (covered).            [unique derivation]
  * |D[query]| >  1  -> ABSTAIN (Mode-B CONFLICT certificate).  [incompatible derivations]
  * |D[query]| == 0  -> ABSTAIN (no connecting path = universe). [absent-relation / underdetermined]

This contract is hallucination-safe by construction: with no connecting path (the
absent-relation case, entities in different components) D is empty, so the engine
NEVER invents a kinship -- it abstains. The naive single-pass baseline uses ONLY the
seed (atomic) edges and one composition step, so it resolves hop-2 chains but abstains
on hop>=3 (the iteration contrast), while the full fixpoint derives the intermediate
composite edges first and resolves the whole chain.
"""
from __future__ import annotations

from collections import defaultdict, deque
from typing import Iterable


class Kinship:
    """Finite kinship composition calculus parsed from the dataset composition table."""

    def __init__(self, comp_table: dict):
        rt = comp_table["relation_types"]
        self.base: list[str] = list(rt.keys())  # 11 abstract relation types
        self.universe = frozenset(self.base)
        self.empty = frozenset()
        self.symmetric_types = set(comp_table["symmetric_types"])  # {'sibling','SO'}
        self.inv: dict[str, str] = {}
        for a, b in comp_table["inverse_pairs"].items():
            self.inv[a] = b
            self.inv[b] = a
        self.composition_rules = comp_table["composition_rules"]
        self.surface_forms = comp_table["surface_forms"]
        self.surface_reverse = comp_table["surface_reverse"]
        self.label_map = comp_table.get("label_map", {})
        self.label_map_reverse = comp_table.get("label_map_reverse", {})
        # ---- total converse over every base type (sound; no empties) ----
        self._conv: dict[str, str] = {}
        for t in self.base:
            if t in self.symmetric_types:
                self._conv[t] = t
            elif t in self.inv:
                self._conv[t] = self.inv[t]
            elif t == "sibling-in-law":
                # brother/sister-in-law are mutual: converse(sibling-in-law)=sibling-in-law.
                self._conv[t] = t
            else:
                self._conv[t] = t  # sound self-converse fallback (never reached for the 11 types)

    # ------------------------------------------------------------------ ops
    def conv_type(self, t: str) -> str:
        return self._conv[t]

    def compose_types(self, t1: str, t2: str):
        """Defined composition rules[t1][t2]=t3, else None (UNDEFINED == 'unknown')."""
        return self.composition_rules.get(t1, {}).get(t2)

    def label(self, s) -> str:
        s = frozenset(s)
        if not s:
            return "EMPTY"
        if s == self.universe:
            return "UNIVERSE"
        return "|".join(t for t in self.base if t in s)

    # ------------------------------------------------------------- surface words
    def surface(self, rel_type: str, gender: str) -> str:
        g = "male" if str(gender).lower().startswith("m") else "female"
        sf = self.surface_forms.get(rel_type)
        if not sf:
            return rel_type
        return sf.get(g, sf.get("male", rel_type))

    def surface_to_type(self, surface_word: str):
        """Return (relation_type, implied_gender) or None for an unknown word."""
        w = str(surface_word).strip().lower()
        rev = self.surface_reverse.get(w)
        if rev is None:
            return None
        return rev[0], rev[1]


# --------------------------------------------------------------------------- #
# Forward least-fixpoint UNION derivation (the sound closure for the finite table)
# --------------------------------------------------------------------------- #
def _seed(kin: Kinship, atomic_edges: list[dict]):
    """Seed D with atomic edges + their converses. Returns (D, nbrs).
    D[(a,b)] = set of types; nbrs[a] = set of directed successors."""
    D: dict = defaultdict(set)
    nbrs: dict = defaultdict(set)

    def add(a, b, t):
        if t not in D[(a, b)]:
            D[(a, b)].add(t)
            nbrs[a].add(b)

    for e in atomic_edges:
        t = e["type"]
        if t not in kin.base:
            continue
        a, b = e["a"], e["b"]
        if a == b:
            continue
        add(a, b, t)
        add(b, a, kin.conv_type(t))
    return D, nbrs


def forward_closure(kin: Kinship, atomic_edges: list[dict], with_prov: bool = False):
    """Forward least-fixpoint union derivation. Returns (D, nbrs, n_fired) or, with
    with_prov, (D, nbrs, n_fired, prov) where prov[(a,c,t3)] = (a,b,c,t1,t2,t3) records
    the FIRST composition that produced type t3 on pair (a,c) (a directed-edge of the
    proof DAG; seed edges map to None).

    D[(a,b)] holds EVERY relation type derivable for the directed pair a->b; closed
    under defined composition + converse. n_fired = number of new type-additions."""
    D, nbrs = _seed(kin, atomic_edges)
    prov: dict = {}
    if with_prov:
        for (a, b), ts in D.items():
            for t in ts:
                prov.setdefault((a, b, t), None)
    Q = deque(D.keys())
    inq = set(D.keys())
    n_fired = 0

    def push(p):
        if p not in inq:
            inq.add(p)
            Q.append(p)

    def emit(a, c, t3, provtuple):
        nonlocal n_fired
        grew = False
        if t3 not in D[(a, c)]:
            D[(a, c)].add(t3)
            nbrs[a].add(c)
            if with_prov:
                prov.setdefault((a, c, t3), provtuple)
            n_fired += 1
            grew = True
        ct3 = kin.conv_type(t3)
        if ct3 not in D[(c, a)]:
            D[(c, a)].add(ct3)
            nbrs[c].add(a)
            if with_prov:
                prov.setdefault((c, a, ct3), (c, a, a, ct3, None, ct3))  # converse marker
        if grew:
            push((a, c)); push((c, a))

    while Q:
        (a, b) = Q.popleft()
        inq.discard((a, b))
        tab = list(D[(a, b)])
        # extend a->b with b->c  =>  a->c
        for c in list(nbrs[b]):
            if c == a:
                continue
            for t1 in tab:
                for t2 in list(D[(b, c)]):
                    t3 = kin.compose_types(t1, t2)
                    if t3 is not None:
                        emit(a, c, t3, (a, b, c, t1, t2, t3))
        # extend z->a with a->b  =>  z->b   (a is the middle)
        for z in list(nbrs[a]):
            if z == b:
                continue
            for t1 in list(D[(z, a)]):
                for t2 in tab:
                    t3 = kin.compose_types(t1, t2)
                    if t3 is not None:
                        emit(z, b, t3, (z, a, b, t1, t2, t3))
    if with_prov:
        return D, nbrs, n_fired, prov
    return D, nbrs, n_fired


def naive_single_pass(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt) -> set:
    """BASELINE: ONE composition pass at the query edge using ONLY seed (atomic) edges.

    R = union over intermediates w of {rules[t1][t2] : t1 in seed(u,w), t2 in seed(w,v)}.
    NO fixpoint, NO derived edges. On a hop-k chain only the hop-2 case has an
    intermediate w with BOTH atomic links to the endpoints, so naive resolves hop-2 but
    derives nothing (-> abstain) on hop>=3."""
    D, nbrs = _seed(kin, atomic_edges)
    R: set = set()
    for w in nbrs.get(qsrc, ()):
        if w in (qsrc, qtgt):
            continue
        if (w, qtgt) in D:
            for t1 in D[(qsrc, w)]:
                for t2 in D[(w, qtgt)]:
                    t3 = kin.compose_types(t1, t2)
                    if t3 is not None:
                        R.add(t3)
    return R


# --------------------------------------------------------------------------- #
# Query wrappers with the Mode-A / Mode-B output contract
# --------------------------------------------------------------------------- #
def _answer_from_set(kin: Kinship, R: set) -> dict:
    R = set(R)
    n = len(R)
    if n == 1:
        t = next(iter(R))
        return {"types": sorted(R), "singleton": True, "answer_type": t,
                "n_derivations": n, "mode_b_conflict": False, "no_path": False}
    if n == 0:
        return {"types": [], "singleton": False, "answer_type": None,
                "n_derivations": 0, "mode_b_conflict": False, "no_path": True}
    # n > 1 : incompatible derivations => Mode-B conflict
    rep = sorted(R, key=lambda t: kin.base.index(t))[0]  # deterministic representative
    return {"types": sorted(R), "singleton": False, "answer_type": rep,
            "n_derivations": n, "mode_b_conflict": True, "no_path": False}


def query_modeA(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt) -> dict:
    """Mode-A forward-closure query. Returns the output-contract decision + n_fired."""
    D, nbrs, n_fired = forward_closure(kin, atomic_edges)
    R = D.get((qsrc, qtgt), set())
    out = _answer_from_set(kin, R)
    out["n_fired"] = n_fired
    return out


def query_naive(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt) -> dict:
    """Naive single-pass query (fresh seed only)."""
    R = naive_single_pass(kin, atomic_edges, qsrc, qtgt)
    return _answer_from_set(kin, R)


def simple_paths_names(atomic_edges: list[dict], qsrc, qtgt, max_paths: int = 3,
                       max_len: int = 12):
    """Up to `max_paths` simple undirected entity paths qsrc..qtgt over the atomic-edge
    graph (feeds Path-of-Thoughts). Returns lists of node names, shortest first."""
    adj: dict = {}
    for e in atomic_edges:
        adj.setdefault(e["a"], set()).add(e["b"])
        adj.setdefault(e["b"], set()).add(e["a"])
    if qsrc not in adj or qtgt not in adj:
        return []
    paths: list[list] = []
    stack = [(qsrc, [qsrc])]
    while stack and len(paths) < max_paths * 4:
        node, path = stack.pop()
        if len(path) > max_len:
            continue
        for nb in sorted(adj.get(node, ())):
            if nb == qtgt:
                paths.append(path + [nb])
            elif nb not in path:
                stack.append((nb, path + [nb]))
    paths.sort(key=len)
    # de-dup
    seen = set(); uniq = []
    for p in paths:
        k = tuple(p)
        if k not in seen:
            seen.add(k); uniq.append(p)
    return uniq[:max_paths]


def derivation_trace(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt,
                     max_steps: int = 60):
    """Reconstruct ONE concrete derivation for (qsrc->qtgt) for the trace-graph:
    which (t1 o t2 -> t3) compositions fire, mirroring the gold backward proof.
    Returns a list of {a,b,c,t1,t2,t3} steps producing the answer type, or [] if the
    query is not a unique-derivation singleton."""
    D, nbrs, _, prov = forward_closure(kin, atomic_edges, with_prov=True)
    target = D.get((qsrc, qtgt), set())
    if len(target) != 1:
        return []
    goal_type = next(iter(target))
    steps = []
    stack = [(qsrc, qtgt, goal_type)]
    seen = set()
    while stack and len(steps) < max_steps:
        key = stack.pop()
        if key in seen:
            continue
        seen.add(key)
        p = prov.get(key)
        if p is None:
            continue  # seed edge (atomic fact) -- a leaf of the proof DAG
        a, b, c, t1, t2, t3 = p
        if t2 is None:
            # converse marker: unfold to the forward edge (b->a : conv(t3))
            stack.append((c, a, kin.conv_type(t3)))
            continue
        steps.append({"a": a, "b": b, "c": c, "t1": t1, "t2": t2, "t3": t3})
        stack.append((a, b, t1))
        stack.append((b, c, t2))
    steps.reverse()
    return steps


## Instantiate the engine from the corpus's composition table

The composition table is shipped **verbatim** inside `metadata.composition_table`. We hand it to
the `Kinship` constructor — exactly the `engine_edge_mapping` the dataset documents
(`{a: source, b: target, type: primitive}`).

In [ ]:
CT = data["metadata"]["composition_table"]
KIN = Kinship(CT)
print("relation types :", KIN.base)
print("converse map   :", KIN._conv)
print("defined comps  :", {k: v for k, v in KIN.composition_rules.items()})

## Round-trip gate (adapted from `verify.py`)

For every document: rebuild the closure input from `atomic_edges`, run `forward_closure`
**unchanged**, and assert the dataset's guarantees on the demo slice:

- **PRESENT `never_annotated`** → on the full KB, `D == {located_in}` (singleton EMIT).
- **PRESENT `held_out`** → **drop the single direct `(source,target)` edge**, then the engine
  *still* derives `located_in` (genuine deduction-required).
- **ABSENT** → `D[(a,b)]` and `D[(b,a)]` are **both EMPTY** (ABSTAIN, both directions).
- **cue-present** → each locally-justifiable edge's surface cue lies in its support span.

In [ ]:
# Round-trip gate over the loaded documents (mirrors verify.py, scoped to MAX_DOCS).
n_rows = 0
n_q_na = q_na_ok = 0              # never_annotated present
n_q_ho = q_ho_ok = ho_inset = 0  # held_out present
n_abs = abs_ok = 0
n_modeb = 0
cue_checked = cue_ok = 0
stratum = Counter()

for d in data["datasets"]:
    for r in d["examples"][:MAX_DOCS]:
        n_rows += 1
        gg = json.loads(r["output"])
        full_edges = [{"a": e["source"], "b": e["target"], "type": e["primitive"]} for e in gg["atomic_edges"]]
        edge_set = {(e["source"], e["target"]) for e in gg["atomic_edges"]}
        dir_edges = {(e["source"], e["target"]) for e in gg["atomic_edges"] if e["primitive"] == PRIM}
        D, nbrs, _ = forward_closure(KIN, full_edges)
        n_modeb += sum(1 for ts in D.values() if len(ts) > 1)
        text = r["input"].lower()
        for e in gg["atomic_edges"]:
            if e["locally_justifiable"]:
                cue_checked += 1
                sp = e["support_span"]
                if e["surface_cue"] and e["surface_cue"] in text[sp[0]:sp[1]]:
                    cue_ok += 1
        for q in gg["query_edges"]:
            s, t = q["source"], q["target"]
            if q["query_kind"] == "held_out":
                n_q_ho += 1; stratum["present_held_out"] += 1
                if (s, t) in edge_set:
                    ho_inset += 1
                # ABLATE the single held-out edge, then the engine must still DEDUCE located_in
                seed_wo = [{"a": e["source"], "b": e["target"], "type": e["primitive"]}
                           for e in gg["atomic_edges"] if (e["source"], e["target"]) != (s, t)]
                Dw, _, _ = forward_closure(KIN, seed_wo)
                R = Dw.get((s, t), set())
                if len(R) == 1 and next(iter(R)) == PRIM:
                    q_ho_ok += 1
            else:
                n_q_na += 1; stratum["present_never_annotated"] += 1
                R = D.get((s, t), set())
                if len(R) == 1 and next(iter(R)) == PRIM:
                    q_na_ok += 1
        for p in gg["absent_relation_pairs"]:
            n_abs += 1; stratum["absent_" + p["reason"]] += 1
            a, b = p["source"], p["target"]
            if not D.get((a, b)) and not D.get((b, a)):     # EMPTY in BOTH directions
                abs_ok += 1

print(f"documents processed: {n_rows}")
print(f"PRESENT never_annotated EMIT : {q_na_ok}/{n_q_na}  (full-KB singleton)")
print(f"PRESENT held_out      EMIT : {q_ho_ok}/{n_q_ho}  (deduced AFTER ablating the direct edge)")
print(f"  held_out edge present in atomic_edges: {ho_inset}/{n_q_ho}")
print(f"ABSENT no-derivation ABSTAIN : {abs_ok}/{n_abs}  (engine derives EMPTY in both dirs)")
print(f"cue-present pass rate         : {cue_ok}/{cue_checked}")
print(f"Mode-B conflicts (annotation cycles): {n_modeb}")

assert q_na_ok == n_q_na, "engine FAILED to reproduce some never_annotated present gold"
assert q_ho_ok == n_q_ho, "engine FAILED to deduce some held_out present gold after ablation"
assert ho_inset == n_q_ho, "some held_out query edge is missing from atomic_edges"
assert abs_ok == n_abs, "some absent pair has a non-empty derivation in some direction"
assert cue_ok == cue_checked, "some locally-justifiable cue not in its support span"
print("\nROUND-TRIP GATE PASSED on the demo subset")

## Human-auditable derivation trace-graph

The point of the corpus: when a containment **is** derivable, the engine emits it **with a
proof** — a chain of `located_in ∘ located_in ⇒ located_in` steps over named places. When it is
**not** derivable, the engine **abstains** instead of inventing a relation. Both are shown below
on the loaded documents.

In [ ]:
# (a) PRESENT held_out: drop the direct edge, derive located_in, reconstruct the trace.
def show_held_out_trace(gg):
    id2surf = {n["entity_id"]: n["surface"] for n in gg["nodes"]}
    for q in gg["query_edges"]:
        if q["query_kind"] != "held_out":
            continue
        s, t = q["source"], q["target"]
        seed_wo = [{"a": e["source"], "b": e["target"], "type": e["primitive"]}
                   for e in gg["atomic_edges"] if (e["source"], e["target"]) != (s, t)]
        D, _, _ = forward_closure(KIN, seed_wo)
        R = D.get((s, t), set())
        steps = derivation_trace(KIN, seed_wo, s, t, max_steps=TRACE_MAX_STEPS)
        print(f"DOC: {gg['doc_id']}")
        print(f"QUERY (direct edge held out): {id2surf[s]}  --located_in?-->  {id2surf[t]}")
        print(f"  engine derivation D = {sorted(R)}   (singleton => EMIT, hallucination-safe)")
        print("  TRACE-GRAPH (each step = a defined located_in o located_in => located_in):")
        for st in steps:
            print(f"    {id2surf[st['a']]} located_in {id2surf[st['b']]}  +  "
                  f"{id2surf[st['b']]} located_in {id2surf[st['c']]}  =>  "
                  f"{id2surf[st['a']]} located_in {id2surf[st['c']]}")
        return True
    return False

for d in data["datasets"]:
    for r in d["examples"][:MAX_DOCS]:
        gg = json.loads(r["output"])
        if any(q["query_kind"] == "held_out" for q in gg["query_edges"]):
            if show_held_out_trace(gg):
                break
    else:
        continue
    break

# (b) ABSENT pair: no derivation path => the engine ABSTAINS (no-derivation certificate).
print("\n" + "-" * 70)
for d in data["datasets"]:
    for r in d["examples"][:MAX_DOCS]:
        gg = json.loads(r["output"])
        if not gg["absent_relation_pairs"]:
            continue
        id2surf = {n["entity_id"]: n["surface"] for n in gg["nodes"]}
        full_edges = [{"a": e["source"], "b": e["target"], "type": e["primitive"]} for e in gg["atomic_edges"]]
        D, _, _ = forward_closure(KIN, full_edges)
        p = gg["absent_relation_pairs"][0]; a, b = p["source"], p["target"]
        print(f"DOC: {gg['doc_id']}")
        print(f"ABSENT QUERY: {id2surf[a]}  <-located_in?->  {id2surf[b]}   (regime: {p['reason']})")
        print(f"  D(a->b) = {sorted(D.get((a, b), set()))}    D(b->a) = {sorted(D.get((b, a), set()))}")
        print("  EMPTY in BOTH directions => engine ABSTAINS (never invents a containment).")
        break
    else:
        continue
    break

## Summary & visualization

Left: how the query/pair strata distribute over the demo documents. Right: the round-trip gate
pass-rate per stratum (all should be 1.0 — the engine reproduces every gold decision).

In [ ]:
# --- Visualization of strata distribution + round-trip gate pass rates ---
labels = list(stratum.keys())
vals = [stratum[k] for k in labels]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(labels, vals, color="#4C72B0")
axes[0].set_title(f"Query / pair strata over {n_rows} demo documents")
axes[0].set_xlabel("count")
for i, v in enumerate(vals):
    axes[0].text(v, i, f" {v}", va="center")

names = ["never_annotated\n(EMIT)", "held_out\n(EMIT)", "absent\n(ABSTAIN)", "cue\npresent"]
oks = [q_na_ok, q_ho_ok, abs_ok, cue_ok]
tots = [n_q_na, n_q_ho, n_abs, cue_checked]
rates = [(o / t if t else 1.0) for o, t in zip(oks, tots)]
axes[1].bar(names, rates, color="#55A868")
axes[1].set_ylim(0, 1.10)
axes[1].set_title("Round-trip gate pass rate (engine vs gold)")
axes[1].set_ylabel("fraction reproduced")
for i, (o, t) in enumerate(zip(oks, tots)):
    axes[1].text(i, rates[i] + 0.02, f"{o}/{t}", ha="center")

plt.tight_layout()
plt.savefig("demo_summary.png", dpi=110)
plt.show()
print("saved demo_summary.png")

# --- full-corpus build stats (for reference; the demo ran a tiny curated slice) ---
bs = data["metadata"]["build_stats"]["re-docred"]
print("\nFull re-docred build (reference, from metadata.build_stats):")
for k in ["n_docs", "present_total", "present_never_annotated", "present_held_out",
          "absent_total", "absent_different_component", "absent_same_component_sibling",
          "mode_b_conflicts", "contradiction_pairs"]:
    print(f"  {k:32s}: {bs[k]}")